In [9]:
import ollama
import json
import re
# -----------------------------
# 1. LLM CALL WITH DELAY SUPPORT
# -----------------------------
def call_llm(user_input):
    prompt = f"""You are a system dynamics expert. Your job is to extract stock-and-flow structures from text descriptions.

Text:
\"\"\"{user_input}\"\"\"

REQUIRED OUTPUT FORMAT - You MUST return valid JSON with ALL of these keys:
{{
  "stocks": ["Name of accumulating quantity"],
  "inflows": ["Name of rate that increases stock"],
  "outflows": ["Name of rate that decreases stock"],
  "auxiliary": ["Name of helper variables"],
  "delays": [
    {{
      "name": "Name of the delayed variable",
      "input": "Variable being delayed",
      "delay_time": 3,
      "delay_type": "first_order",
      "initial_value": 0
    }}
  ],
  "equations": {{
    "stock": "INTEG(inflow_name - outflow_name, initial_value)",
    "inflow": "mathematical formula",
    "outflow": "mathematical formula or DELAY_VAR(input, delay_time, initial)",
    "auxiliary": "formula"
  }},
  "units": {{}}
}}

RULES:
1. STOCKS accumulate over time (e.g., "Views", "Water Level", "Population")
2. INFLOWS are rates that ADD to stocks (e.g., "New Views", "Filling Rate")
3. OUTFLOWS are rates that SUBTRACT from stocks (e.g., "View Decay", "Leakage")
4. AUXILIARY are helper variables that aren't stocks or flows
5. DELAYS capture time-lagged effects between variables
   - Use "first_order" for smooth exponential delays (SMOOTH)
   - Use "third_order" for more realistic material/pipeline delays (DELAY3)
   - Use "pipeline" for fixed transit-time delays (DELAY FIXED)
   - Add a delay when the text says: "after some time", "lag", "takes time", "delayed effect",
     "pipeline", "delivery time", "adjustment time", "time to build", "response delay"
6. Each stock MUST use INTEG(inflow - outflow, initial_value)
7. Delayed variables use SMOOTH(), DELAY3(), or DELAY FIXED() in equations
8. UNITS (VERY IMPORTANT):
   - Every variable MUST have a unit
   - Stocks: base unit (e.g., people, houses, dollars)
   - Flows: stock/time (e.g., people/month)
   - Delays: time (e.g., month)
   - Probabilities: dimensionless
   - Rates: 1/time
9. Every variable name that appears in any equation MUST be declared in exactly one of: stocks, inflows, outflows, or auxiliary. If you write a variable in an equation, it MUST appear in the lists. No "ghost" variables allowed. [STRICT]
10. Return ONLY the JSON object, no markdown, no code blocks, no explanations

DELAY EQUATION FORMATS:
- First-order (smooth):  SMOOTH(input_var, delay_time, initial_value)
- Third-order (realistic): DELAY3(input_var, delay_time, initial_value)
- Pipeline (fixed):      DELAY FIXED(input_var, delay_time, initial_value)

EXAMPLE WITH DELAY:
Input: "New hires take 3 months to become productive. Hiring rate depends on the workforce gap."
Output:
{{
  "stocks": ["Workforce"],
  "inflows": ["Onboarding Completions"],
  "outflows": ["Attrition"],
  "auxiliary": ["Desired Workforce", "Workforce Gap", "Attrition Rate"],
  "delays": [
    {{
      "name": "Onboarding Completions",
      "input": "Hiring Rate",
      "delay_time": 3,
      "delay_type": "third_order",
      "initial_value": 0
    }}
  ],
  "equations": {{
    "Workforce": "INTEG(Onboarding Completions - Attrition, 100)",
    "Hiring Rate": "MAX(0, Workforce Gap / 1)",
    "Onboarding Completions": "DELAY3(Hiring Rate, 3, 0)",
    "Attrition": "Workforce * Attrition Rate",
    "Desired Workforce": "150",
    "Workforce Gap": "Desired Workforce - Workforce",
    "Attrition Rate": "0.05"
  }},
  "units": {{
    "Workforce": "People",
    "Onboarding Completions": "People/Month",
    "Attrition": "People/Month",
    "Hiring Rate": "People/Month",
    "Desired Workforce": "People",
    "Workforce Gap": "People",
    "Attrition Rate": "1/Month"
  }}
}}

EXAMPLE WITHOUT DELAY (for comparison):
Input: "When a post gets more views, people share it more, bringing in even more views."
Output:
{{
  "stocks": ["Post Views"],
  "inflows": ["New Views from Shares"],
  "outflows": ["View Decay"],
  "auxiliary": ["Share Probability", "Decay Rate"],
  "delays": [],
  "equations": {{
    "Post Views": "INTEG(New Views from Shares - View Decay, 0)",
    "New Views from Shares": "Post Views * Share Probability",
    "View Decay": "Post Views * Decay Rate",
    "Share Probability": "0.1",
    "Decay Rate": "0.05"
  }},
  "units": {{
    "Post Views": "Views",
    "New Views from Shares": "Views/Day",
    "View Decay": "Views/Day",
    "Share Probability": "Dimensionless",
    "Decay Rate": "1/Day"
  }}
}}

Now analyze the given text and return JSON only.
"""
    try:
        response = ollama.chat(
            model='gemma4',
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

    except Exception as e:
        print("LLM call failed:", e)
        return ""
    
def parse_json(response_text):
    try:
        # Try direct parsing first
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract JSON from messy output (handles markdown code blocks)
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(1))
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract raw JSON object
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except json.JSONDecodeError as e:
        print("JSON parsing error:", e)
    
    return None

def validate(data):
    errors = []
 
    if not data:
        errors.append("No data parsed")
        return errors
 
    # Check for required keys (using plural forms to match prompt)
    if not data.get("stocks") or len(data.get("stocks", [])) == 0:
        errors.append("Missing stocks")
 
    if not data.get("inflows") or len(data.get("inflows", [])) == 0:
        errors.append("No inflows defined")
 
    if not data.get("outflows") or len(data.get("outflows", [])) == 0:
        errors.append("No outflows defined")
 
    if not data.get("equations") or len(data.get("equations", {})) == 0:
        errors.append("No equations provided")

    return errors

def generate_mdl(data):
    lines = []

    # Header
    lines.append("{UTF-8}")

    equations = data.get("equations", {})
    units = data.get("units", {})

    def format_var(name):
        return name.lower()

    def format_unit(unit):
        """Convert to Vensim style (Month instead of month)"""
        if not unit:
            return "dimensionless"
        return unit.replace("month", "Month").replace("per", "/")


    # Generate variables
    for var, eq in equations.items():
        var_name = format_var(var)
        eq_formatted = eq.strip()

        unit = format_unit(units.get(var, "dimensionless"))

        lines.append(f"{var_name} = {eq_formatted}")
        lines.append(f"~ {unit}")
        lines.append(f"~ |")
        lines.append("")

    # Control section
    lines.append("********************************************************")
    lines.append(".Control")
    lines.append("********************************************************~")
    lines.append("Simulation Control Parameters")
    lines.append("|")
    lines.append("")
    lines.append("FINAL TIME  = 100")
    lines.append("~ Month")
    lines.append("~ The final time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("INITIAL TIME  = 0")
    lines.append("~ Month")
    lines.append("~ The initial time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("SAVEPER  =")
    lines.append("        TIME STEP")
    lines.append("~ Month")
    lines.append("~ The frequency with which output is stored.")
    lines.append("|")
    lines.append("")
    lines.append("TIME STEP  = 1")
    lines.append("~ Month")
    lines.append("~ The time step for the simulation.")
    lines.append("|")

    return "\n".join(lines)


# -----------------------------
# 5. FULL PIPELINE
# -----------------------------
def run_pipeline(user_input):
    print("🔹 Input:", user_input)
    print("\n" + "="*70)
 
    raw = call_llm(user_input)
    print("\n🔹 Raw LLM Output:")
    print(raw)
    print("="*70)
 
    parsed = parse_json(raw)
    print("\n🔹 Parsed JSON (before extraction):")
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print("Failed to parse JSON")

 
    errors = validate(parsed)
 
    if errors:
        print("\n❌ Validation Errors:")
        for error in errors:
            print(f"  - {error}")
        return None
 
    mdl = generate_mdl(parsed)
    print("\n✅ MDL Output:")
    print(mdl)
    print("="*70)
 
    return parsed, mdl

In [ ]:
run_pipeline("An increase in complaints leads to an increase in staff count, " \
"as the organization hires more employees to handle the issues. " \
"As staff count increases, mentoring workload for senior staff also increases, since more new employees require guidance and training." \
" However, increased mentoring workload reduces senior availability (negative relationship), as senior staff have less time for other tasks." \
" Lower senior availability leads to longer response time, meaning it takes more time to address customer issues. " \
"As response time increases, complaints increase (positive relationship), completing the reinforcing loop. " \
"There is a delay between staff count and mentoring workload, representing the time it takes for new hires to require mentoring.")

🔹 Input: Orders placed by customers increase the inventory after a shipping delay of 5 days. Sales decrease the inventory immediately.


🔹 Raw LLM Output:
{
  "stocks": ["Inventory"],
  "inflows": ["Orders Received (Delayed)"],
  "outflows": ["Sales"],
  "auxiliary": ["Orders Placed"],
  "delays": [
    {
      "name": "Orders Received (Delayed)",
      "input": "Orders Placed",
      "delay_time": 5,
      "delay_type": "pipeline",
      "initial_value": 0
    }
  ],
  "equations": {
    "Inventory": "INTEG(Orders Received (Delayed) - Sales, 0)",
    "Orders Received (Delayed)": "DELAY FIXED(Orders Placed, 5, 0)",
    "Sales": "Inventory * 0.01",
    "Orders Placed": "Customers * 1"
  },
  "units": {
    "Inventory": "Units",
    "Orders Received (Delayed)": "Units/Day",
    "Sales": "Units/Day",
    "Orders Placed": "Units/Day",
    "Customers": "People"
  }
}

🔹 Parsed JSON (before extraction):
{
  "stocks": [
    "Inventory"
  ],
  "inflows": [
    "Orders Received (Delayed)"
  ],


({'stocks': ['Inventory'],
  'inflows': ['Orders Received (Delayed)'],
  'outflows': ['Sales'],
  'auxiliary': ['Orders Placed'],
  'delays': [{'name': 'Orders Received (Delayed)',
    'input': 'Orders Placed',
    'delay_time': 5,
    'delay_type': 'pipeline',
    'initial_value': 0}],
  'equations': {'Inventory': 'INTEG(Orders Received (Delayed) - Sales, 0)',
   'Orders Received (Delayed)': 'DELAY FIXED(Orders Placed, 5, 0)',
   'Sales': 'Inventory * 0.01',
   'Orders Placed': 'Customers * 1'},
  'units': {'Inventory': 'Units',
   'Orders Received (Delayed)': 'Units/Day',
   'Sales': 'Units/Day',
   'Orders Placed': 'Units/Day',
   'Customers': 'People'}},
 '{UTF-8}\ninventory = INTEG(Orders Received (Delayed) - Sales, 0)\n~ Units\n~ |\n\norders received (delayed) = DELAY FIXED(Orders Placed, 5, 0)\n~ Units/Day\n~ |\n\nsales = Inventory * 0.01\n~ Units/Day\n~ |\n\norders placed = Customers * 1\n~ Units/Day\n~ |\n\n********************************************************\n.Control\n**